# Joshi Part 7: Black-Scholes & Greeks (Python)

Closed-form Black-Scholes pricing, all five Greeks, and put-call parity.

In [ ]:
import math
import numpy as np

def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def normal_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def d1_d2(S, K, r, sigma, T):
    vol_sqrt_t = sigma * math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    return d1, d2

In [ ]:
def bs_call(S, K, r, sigma, T):
    d1, d2 = d1_d2(S, K, r, sigma, T)
    return S * normal_cdf(d1) - K * math.exp(-r * T) * normal_cdf(d2)

def bs_put(S, K, r, sigma, T):
    d1, d2 = d1_d2(S, K, r, sigma, T)
    return K * math.exp(-r * T) * normal_cdf(-d2) - S * normal_cdf(-d1)

# ATM option: S=100, K=100, r=5%, vol=20%, T=1
S, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

c = bs_call(S, K, r, sigma, T)
p = bs_put(S, K, r, sigma, T)
parity = c - p - (S - K * math.exp(-r * T))

print(f"Call price:  {c:.6f}")
print(f"Put price:   {p:.6f}")
print(f"Put-call parity residual: {parity:.2e}")

## Greeks

In [ ]:
def call_delta(S, K, r, sigma, T):
    d1, _ = d1_d2(S, K, r, sigma, T)
    return normal_cdf(d1)

def gamma(S, K, r, sigma, T):
    d1, _ = d1_d2(S, K, r, sigma, T)
    return normal_pdf(d1) / (S * sigma * math.sqrt(T))

def vega(S, K, r, sigma, T):
    d1, _ = d1_d2(S, K, r, sigma, T)
    return S * normal_pdf(d1) * math.sqrt(T)

def call_theta(S, K, r, sigma, T):
    d1, d2 = d1_d2(S, K, r, sigma, T)
    t1 = -S * normal_pdf(d1) * sigma / (2 * math.sqrt(T))
    t2 = -r * K * math.exp(-r * T) * normal_cdf(d2)
    return t1 + t2

def call_rho(S, K, r, sigma, T):
    _, d2 = d1_d2(S, K, r, sigma, T)
    return K * T * math.exp(-r * T) * normal_cdf(d2)

print(f"{'Greek':<10} {'Value':>12}")
print("-" * 24)
print(f"{'Delta':<10} {call_delta(S, K, r, sigma, T):>12.6f}")
print(f"{'Gamma':<10} {gamma(S, K, r, sigma, T):>12.6f}")
print(f"{'Vega':<10} {vega(S, K, r, sigma, T):>12.6f}")
print(f"{'Theta':<10} {call_theta(S, K, r, sigma, T):>12.6f}")
print(f"{'Rho':<10} {call_rho(S, K, r, sigma, T):>12.6f}")

## Spot vs Delta surface

In [ ]:
spots = np.linspace(60, 140, 50)
deltas = [call_delta(s, 100, 0.05, 0.2, 1.0) for s in spots]
gammas = [gamma(s, 100, 0.05, 0.2, 1.0) for s in spots]

print("Spot → Delta and Gamma for K=100, vol=20%, T=1:")
for s, d, g in list(zip(spots, deltas, gammas))[::10]:
    print(f"  S={s:6.1f}  Δ={d:.4f}  Γ={g:.6f}")